# Revised Experiment 2 pilot and GPU qualification

Set `SOURCE_SHA` to the reviewed implementation commit before pushing. Default mode qualifies four full-size/full-10k-raw-step branches, full plasticity, learning-branch restart and measured solo memory before admitting a concurrent stress on each detected GPU. Reference qualification training is 10k raw steps at one checkpoint, so this is engineering evidence only. Pilot mode schedules the complete twenty-run/million-step/fifteen-checkpoint design. Outputs/logs stay in `/kaggle/working/exp2-revised`; preserve them between Kaggle sessions. No GPU model is assumed. Concurrency never exceeds two independent processes per GPU and falls back to one without matching measured evidence. Runtime dependencies follow the previously working Kaggle setup, including its Python 3.13 compatibility substitutions; this differs from the pinned local CPU environment.


In [ ]:
import os
os.environ.update({"JAX_PLATFORMS": "cuda,cpu", "MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl", "EGL_PLATFORM": "surfaceless", "MUJOCO_EGL_DEVICE_ID": "0", "WANDB_MODE": "disabled", "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "XLA_PYTHON_CLIENT_PREALLOCATE": "false"})
import signal
def _qualification_timeout(signum, frame):
    raise TimeoutError("Experiment 2 pilot GPU qualification exceeded its 11-hour limit")
signal.signal(signal.SIGALRM, _qualification_timeout)
signal.alarm(11 * 60 * 60)
import platform
import subprocess
import sys
from pathlib import Path

SOURCE_SHA = os.environ.get("EXP2_SOURCE_SHA", "a7b229335b9737ab1ea1fe1b9f428bbc3ea26a6a")
SOURCE_BRANCH = "codex/exp2-online-prototype-20261107"
REPO = Path("/kaggle/temp/scaling-drl-research")
REPO.parent.mkdir(parents=True, exist_ok=True)
# Kaggle's internet toggle is required for both source and dependency downloads.
import socket
import time
import tempfile
import shutil

def wait_for_dns(host, attempts=6, delay=10):
    for attempt in range(attempts):
        try:
            socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            print("DNS available:", host)
            return
        except socket.gaierror as exc:
            print(f"DNS attempt {attempt + 1}/{attempts} failed for {host}: {exc}", flush=True)
            if attempt + 1 < attempts:
                time.sleep(delay)
    raise RuntimeError(f"Kaggle cannot resolve {host}. Check that Internet is enabled in notebook settings; if it is, retry after Kaggle networking recovers.")

for host in ("github.com", "pypi.org", "files.pythonhosted.org"):
    wait_for_dns(host)

if not REPO.exists():
    for attempt in range(3):
        with tempfile.TemporaryDirectory(prefix="exp2-clone-", dir=REPO.parent) as tmp:
            destination = Path(tmp) / "source"
            result = subprocess.run(["git", "clone", "--quiet", "--single-branch", "--branch", SOURCE_BRANCH, "https://github.com/kaves29/scaling-drl-research.git", str(destination)], text=True, capture_output=True)
            if result.returncode == 0:
                shutil.move(str(destination), str(REPO))
                break
            print(f"Git clone attempt {attempt + 1}/3 failed: {result.stderr.strip()}", flush=True)
        if attempt < 2:
            time.sleep(10)
    else:
        raise RuntimeError("Could not clone the pinned source after three attempts; check Kaggle internet access and GitHub availability.")
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == SOURCE_SHA
requirements = (REPO / "main/requirements.txt").read_text()
install_dm_control_without_optional_mazes = sys.version_info >= (3, 13)
if install_dm_control_without_optional_mazes:
    replacements = {"numpy==1.26.4": "numpy==2.1.3", "pandas==2.1.4": "pandas==2.2.3", "matplotlib==3.9.0": "matplotlib==3.9.2", "Shimmy==2.0.0": "Shimmy==1.3.0"}
    for old, new in replacements.items():
        assert requirements.count(old) == 1, f"Expected exactly one {old} pin"
        requirements = requirements.replace(old, new)
    for unused_pin in ("labmaze==1.0.6", "dm_control==1.0.38"):
        assert requirements.count(unused_pin) == 1, f"Expected exactly one {unused_pin} pin"
        requirements = requirements.replace(unused_pin + "\n", "")
runtime_requirements = Path("/kaggle/temp/exp2-runtime-requirements.txt")
runtime_requirements.write_text(requirements)
print("Python runtime:", platform.python_version(), "; installing runtime requirements:", runtime_requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(runtime_requirements)], check=True)
if install_dm_control_without_optional_mazes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "dm_control==1.0.38"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda13-plugin", "jax-cuda13-pjrt"], check=True, stdout=subprocess.DEVNULL)
    print("Removed Kaggle CUDA 13 JAX plugins to prevent conflicts with the pinned CUDA 12 backend.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]==0.4.34"], check=True)

compat = Path("/kaggle/temp/exp2-python-compat")
compat.mkdir(exist_ok=True)
(compat / "sitecustomize.py").write_text("import numpy as np\nif not hasattr(np, 'issctype'):\n    np.issctype = lambda rep: isinstance(rep, type) and issubclass(rep, np.generic)\n")
os.environ["PYTHONPATH"] = str(compat) + os.pathsep + str(REPO / "main")
# Engineering qualification logs are retained for later W&B sync.
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_DIR"] = "/kaggle/working"
print("Qualification W&B logs will be saved offline for later synchronization.")


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

# Version 4 output is attached as a read-only notebook input; copy it to a
# writable directory before resuming. Source identity checks remain enabled.
inputs = Path("/kaggle/input")
candidates = list(inputs.rglob("qualification_sequential_gpu0_reference_small"))
if not candidates:
    import tarfile
    archives = list(inputs.rglob("*.tar"))
    assert len(archives) == 1, f"Expected one checkpoint archive; found {len(archives)}"
    extracted = Path("/kaggle/temp/exp2-recovery")
    extracted.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archives[0]) as archive:
        archive.extractall(extracted, filter="data")
    candidates = list(extracted.rglob("qualification_sequential_gpu0_reference_small"))
assert len(candidates) == 1, f"Expected one interrupted checkpoint; found {len(candidates)}"
source = candidates[0]
previous = source.parent / "qualification_solo_gpu0_reference_large" / "qualification_result.json"
large = json.loads(previous.read_text())
assert large["source_commit"] == SOURCE_SHA and large["full_branch_horizon"] and large["plasticity_probe"] and large["restart"]
output = Path("/kaggle/working/exp2-resumed-small")
if not output.exists():
    shutil.copytree(source, output)
assert (output / "state" / "LATEST").is_file()
print("Restoring interrupted small-reference qualification; verified previous large-reference receipt", flush=True)
command = [sys.executable, "-m", "experiments.exp12.revised_exp2_qualification", "--regime", "reference_small", "--output", str(output)]
with (output / "resume-process.log").open("a") as log:
    subprocess.run(command, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", MUJOCO_EGL_DEVICE_ID="0"), stdout=log, stderr=subprocess.STDOUT, check=True)
small = json.loads((output / "qualification_result.json").read_text())
assert small["source_commit"] == SOURCE_SHA and small["full_branch_horizon"] and small["plasticity_probe"] and small["restart"]
print("PASS: both reference regimes completed full branch/probe/restart qualification; no two-jobs-per-GPU qualification", flush=True)
(output / "combined_qualification.json").write_text(json.dumps({"large": large, "small": small, "jobs_per_gpu": 1}, indent=2))
